# Import

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from tqdm.notebook import tqdm

In [2]:
import spacy
nlp = spacy.load("de_core_news_md")

In [3]:
reviews = pd.read_csv("../../corpus/lovelybooks_reviews.csv").drop(columns=["Unnamed: 0"]).drop_duplicates(subset='review_id')

display(reviews.shape[0])
display(reviews['reviewer'].nunique())

/var/folders/51/fcpkm27j0y9d8nsp093jt53r0000gn/T/ipykernel_46236/3492066630.py:1: DtypeWarning: Columns (0: content_short, 1: date_scraped) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv("../../corpus/lovelybooks_reviews.csv").drop(columns=["Unnamed: 0"]).drop_duplicates(subset='review_id')


2860308

72603

# Functions

In [4]:
def get_adjectives_for_token(token):
    results = []

    def find_coordinated_adjs(parent_token, adj_type):
        """Recursively find all chained adjectives, passing through conjunctions."""
        found = []
        for child in parent_token.children:
            # Case A: The child is a coordinated adjective/adverb directly
            if child.dep_ in {"cj", "cd"} and child.pos_ in {"ADJ", "ADV"}:
                found.append((child.lemma_, adj_type))
                found.extend(find_coordinated_adjs(child, adj_type))
            
            # Case B: The child is a conjunction word (like 'und', 'oder')
            # We don't extract its lemma, but we dig into its children to find the final adjective
            elif child.pos_ in {"CCONJ", "KON", "CC"} or child.dep_ == "cd":
                found.extend(find_coordinated_adjs(child, adj_type))
                
        return found

    # --- 1. Attributive adjectives ---
    for child in token.children:
        if child.dep_ in {"amod", "nk"} and child.pos_ == "ADJ":
            results.append((child.lemma_, "attributive"))
            results.extend(find_coordinated_adjs(child, "attributive"))

    # --- 2. Predicative adjectives ---
    PREDICATIVE_DEPS = {"acomp", "pd", "oc"}
    COPULAS = {"sein", "werden", "bleiben", "scheinen", "heißen"}

    if token.dep_ in {"nsubj", "sb", "oa", "da", "og"}:
        verb = token.head
        verb_is_copula = verb.lemma_ in COPULAS
        for child in verb.children:
            if child.dep_ in PREDICATIVE_DEPS and child.pos_ in {"ADJ", "ADV"}:
                # Only accept "pd" if the verb is a copula
                if child.dep_ == "pd" and not verb_is_copula:
                    continue
                results.append((child.lemma_, "predicative"))
                results.extend(find_coordinated_adjs(child, "predicative"))

    results = [(adj.lower(), adj_type) for adj, adj_type in results]
    
    return results

In [5]:
test_docs = [
    "Die spannende und fantastische Literatur ist Aufregend, anschaulich und genial.",
    "Klar doch, es gibt gute Literatur dazu.",
    "Wie entsteht Literatur?",
    "Ich finde es immer wieder faszinierend, wie japanische Literatur so ruhig und doch so fesselnd sein kann.",
    "Ich hatte mich sehr auf Hatokos wunderbarer Schreibwarenladen gefreut, da ich sehr gerne japanische und koreanische Literatur lese, da diese auf mich sehr oft einen bestimmten Lese-Zauber hat, den ich mir auch bei diesem Buch erhofft hatte.",
    "Umso schöner, weil das wieder einmal verdeutlicht, wie viel Literatur bewirken kann: Rezipierende erfahren und erleben ohne selbst inkludiert zu sein.",
    "So ist gute Literatur."
]

for doc in test_docs:
    print(f"\n{doc}")
    doc = nlp(doc)
    for token in doc:
        if token.text == 'Literatur':
            print(get_adjectives_for_token(token))


Die spannende und fantastische Literatur ist Aufregend, anschaulich und genial.
[('spannend', 'attributive'), ('fantastisch', 'attributive'), ('aufregend', 'predicative'), ('anschaulich', 'predicative'), ('genial', 'predicative')]

Klar doch, es gibt gute Literatur dazu.
[('gut', 'attributive')]

Wie entsteht Literatur?
[]

Ich finde es immer wieder faszinierend, wie japanische Literatur so ruhig und doch so fesselnd sein kann.
[('japanisch', 'attributive')]

Ich hatte mich sehr auf Hatokos wunderbarer Schreibwarenladen gefreut, da ich sehr gerne japanische und koreanische Literatur lese, da diese auf mich sehr oft einen bestimmten Lese-Zauber hat, den ich mir auch bei diesem Buch erhofft hatte.
[('japanisch', 'attributive'), ('koreanisch', 'attributive')]

Umso schöner, weil das wieder einmal verdeutlicht, wie viel Literatur bewirken kann: Rezipierende erfahren und erleben ohne selbst inkludiert zu sein.
[]

So ist gute Literatur.
[('gut', 'attributive')]


In [6]:
def get_adjectives_from_df(df, sentence_filter_strings = ['litera'], relevant_lemmata = ['literatur'], relevant_pos = ['NOUN'], verbose=True):
    results = []

    for i, row in tqdm(df.reset_index().iterrows(), total=len(df)):
        doc = nlp(row["content"])

        # identify sentences with filter strings
        for sent in doc.sents:
            if any(s.lower() in sent.text.lower() for s in sentence_filter_strings):

                # identify relevant lemmata/pos combinations
                for token in sent:
                    lemma = token.lemma_.lower()
                    if lemma in relevant_lemmata and token.pos_ in relevant_pos:

                        # get adjectives for token
                        token_adjectives = get_adjectives_for_token(token)

                        # add adjective tuples to results
                        for adjective_tuple in token_adjectives:
                            adjective_tuple = adjective_tuple+(sent.text, row["url"])
                            results.append(adjective_tuple)

        if verbose and i > 0 and i % 1000 == 0:
            print(f"{i} reviews analyzed")
            print(f"{len(results)} adjectives found")
            df = pd.DataFrame(results, columns=['adjective', 'type', 'sentence', 'url'])
            display(df['adjective'].value_counts().head(10))

    df = pd.DataFrame(results, columns=['adjective', 'type', 'sentence', 'url'])
    return df

In [7]:
def calculate_llr(row, c1, c2, col_a, col_b):
    o11 = row[f"count_{col_a}"]
    o12 = row[f"count_{col_b}"]
    o21 = c1 - o11
    o22 = c2 - o12
    
    n = c1 + c2
    
    # Expected values for each cell in the contingency table
    e11 = (o11 + o12) * c1 / n
    e12 = (o11 + o12) * c2 / n
    e21 = (o21 + o22) * c1 / n
    e22 = (o21 + o22) * c2 / n
    
    # Helper to calculate individual likelihood parts, avoiding log(0)
    def log_part(o, e):
        return o * np.log(o / e) if o > 0 else 0
    
    # G2 formula: 2 * sum(O * ln(O/E))
    g2 = 2 * (log_part(o11, e11) + log_part(o12, e12) + 
              log_part(o21, e21) + log_part(o22, e22))
    
    # Direction: make it positive if a-dominant, negative if b-dominant
    if (o11 / c1) < (o12 / c2):
        g2 = -g2
        
    return g2

In [8]:
def create_results_table (df, col_a, col_b):
    c1 = df[f"count_{col_a}"].sum() 
    c2 = df[f"count_{col_b}"].sum()   

    df[f'{col_a}_{col_b}_reldiff'] = df[f"rel_{col_a}"]-df[f"rel_{col_b}"]
    df[f'{col_a}_{col_b}_logodds'] = np.log((df[f"rel_{col_a}"] + 0.1) / (df[f"rel_{col_b}"] + 0.1))
    df[f'{col_a}_{col_b}_llr'] = df.apply(calculate_llr, axis=1, args=(c1, c2, col_a, col_b))

    return df

# Get Adjectives

## Literature

In [9]:
literature_reviews = reviews.query("content.str.contains('literatur', case=False, regex=False, na=False)").copy()
literature_reviews.shape[0]

70387

In [ ]:
literature_df = get_adjectives_from_df(
    literature_reviews,
    sentence_filter_strings = ['literatur'],
    relevant_lemmata = ['literatur'],
    relevant_pos = ['NOUN'],
    verbose=False,
)

In [ ]:
literature_df.to_csv("../resources/adjectives_literature.csv")

## Book

In [10]:
book_reviews = reviews.query("content.str.contains('buch|bücher', case=False, regex=True, na=False)").copy()
book_reviews.shape[0]

2271847

In [ ]:
book_df = get_adjectives_from_df(
    book_reviews.sample(n=50000),
    sentence_filter_strings=['buch', 'bücher'],
    relevant_lemmata=['buch'],
    relevant_pos=['NOUN'],
    verbose=False,
)

In [ ]:
book_df.to_csv("../resources/adjectives_book.csv")

## Text

In [11]:
text_reviews = reviews.query("content.str.contains('text', case=False, regex=True, na=False)").copy()
text_reviews.shape[0]

430878

In [ ]:
text_df = get_adjectives_from_df(
    text_reviews.sample(n=200000),
    sentence_filter_strings=['text'],
    relevant_lemmata=['text'],
    relevant_pos=['NOUN'],
    verbose=False,
)

In [ ]:
text_df.to_csv("../resources/adjectives_text.csv")

# Create Results Table

In [12]:
literature_df = pd.read_csv("../resources/adjectives_literature.csv", index_col=[0]).query("adjective.notna()")
book_df = pd.read_csv("../resources/adjectives_book.csv", index_col=[0]).query("adjective.notna()")
text_df = pd.read_csv("../resources/adjectives_text.csv", index_col=[0]).query("adjective.notna()")

print(literature_df.shape[0])
print(book_df.shape[0])
print(text_df.shape[0])

book_df = book_df.sample(n=literature_df.shape[0])
text_df = text_df.sample(n=literature_df.shape[0])

20670
27351
24140


In [13]:
literature_df.head()

,adjective,type,url
0,erotisch,attributive,https://www.lovelybooks.de/autor/Nina-George/D...
1,dazugehörig,attributive,https://www.lovelybooks.de/autor/Nina-George/D...
2,klassisch,attributive,https://www.lovelybooks.de/autor/Nina-George/D...
3,normal,attributive,https://www.lovelybooks.de/autor/Emily-Dunlay/...
4,tiefgründig,attributive,https://www.lovelybooks.de/autor/Liane-Moriart...


In [14]:
literature_vc = literature_df['adjective'].value_counts().to_frame()
literature_vc.columns = ['count_literature']
literature_vc['rel_literature'] = literature_vc['count_literature']/literature_vc['count_literature'].sum()

book_vc = book_df['adjective'].value_counts().to_frame()
book_vc.columns = ['count_book']
book_vc['rel_book'] = book_vc['count_book']/book_vc['count_book'].sum()

text_vc = text_df['adjective'].value_counts().to_frame()
text_vc.columns = ['count_text']
text_vc['rel_text'] = text_vc['count_text']/text_vc['count_text'].sum()

In [15]:
results = (
    literature_vc
    .merge(book_vc, how='outer', on='adjective')
    .merge(text_vc, how='outer', on='adjective')
    .fillna(0)
)

results['count_booktext'] = results['count_book'] + results['count_text']
results['rel_booktext'] = results['count_booktext']/results['count_booktext'].sum()
results = create_results_table(results, 'literature', 'booktext')

In [16]:
results.sort_values(by='count_literature', ascending=False).head()

,count_literature,rel_literature,count_book,rel_book,count_text,rel_text,count_booktext,rel_booktext,literature_booktext_reldiff,literature_booktext_logodds,literature_booktext_llr
adjective,,,,,,,,,,,
englisch,1316.0,0.063667,14.0,0.000677,77.0,0.003725,91.0,0.002201,0.061466,0.470891,2344.632738
groß,944.0,0.045670,30.0,0.001451,72.0,0.003483,102.0,0.002467,0.043203,0.351800,1514.611108
anspruchsvoll,916.0,0.044315,30.0,0.001451,100.0,0.004838,130.0,0.003145,0.041171,0.335869,1356.670466
erotisch,899.0,0.043493,16.0,0.000774,11.0,0.000532,27.0,0.000653,0.042840,0.354606,1778.902989
weiterführend,858.0,0.041509,1.0,0.000048,3.0,0.000145,4.0,0.000097,0.041413,0.346229,1861.577509


In [17]:
display(results[[
    'literature_booktext_reldiff',
    'literature_booktext_logodds',
    'literature_booktext_llr'
]].corr())

,literature_booktext_reldiff,literature_booktext_logodds,literature_booktext_llr
literature_booktext_reldiff,1.000000,0.996569,0.982331
literature_booktext_logodds,0.996569,1.000000,0.977216
literature_booktext_llr,0.982331,0.977216,1.000000


# Show Results

In [18]:
results[['count_literature', 'count_book', 'count_text', 'count_booktext']].sum()

count_literature    20670.0
count_book          20670.0
count_text          20670.0
count_booktext      41340.0
dtype: float64

In [19]:
top_pos = results.nlargest(10, 'literature_booktext_llr')
top_neg = results.nsmallest(10, 'literature_booktext_llr')

diverging = pd.concat([top_neg, top_pos])
diverging = diverging.sort_values('literature_booktext_llr')

diverging.index = diverging.index.to_series().replace({
    "erster" : "erster [first]",
    "toll" : "toll [terrific]",
    "ganz" : "ganz [whole]",
    "anderer" : "anderer [other]",
    "spannend" : "spannend [exciting]",
    "zweiter" : "zweiter [second]",
    "nächster" : "nächster [next]",
    "schön" : "schön [beautiful]",
    "letzter" : "letzter [last]",
    "gesamt" : "gesamt [complete]",
    "zeitgenössisch" : "zeitgenössisch [contemporary]",
    "amerikanisch" : "amerikanisch [American]",
    "hoch" : "hoch [high]",
    "klassisch" : "klassisch [classic]",
    "deutsch" : "deutsch [German]",
    "anspruchsvoll" : "anspruchsvoll [sophisticated]",
    "groß" : "groß [great]",
    "erotisch" : "erotisch [erotic]",
    "weiterführend" : "weiterführend [further]",
    "englisch" : "englisch [English]",
    "japanisch" : "japanisch [Japanese]",
    "neu" : "neu [new]",
    "kurz" : "kurz [short]",
    "historisch" : "historisch [historical]",
    "lang" : "lang [long]",
    "klein" : "klein [small]",
    "kindgerecht" : "kindgerecht [suitable for children]",
    "informativ" : "informativ [informative]",
    "verständlich" : "verständlich [comprehensible]",
})

In [20]:
diverging

,count_literature,rel_literature,count_book,rel_book,count_text,rel_text,count_booktext,rel_booktext,literature_booktext_reldiff,literature_booktext_logodds,literature_booktext_llr
adjective,,,,,,,,,,,
kurz [short],4.0,0.000194,103.0,0.004983,3003.0,0.145283,3106.0,0.075133,-0.074940,-0.558442,-2547.081587
erster [first],2.0,0.000097,1544.0,0.074698,69.0,0.003338,1613.0,0.039018,-0.038921,-0.328465,-1302.991212
verständlich [comprehensible],5.0,0.000242,24.0,0.001161,1159.0,0.056072,1183.0,0.028616,-0.028374,-0.249248,-916.907033
ganz [whole],11.0,0.000532,807.0,0.039042,156.0,0.007547,963.0,0.023295,-0.022762,-0.204099,-691.817951
toll [terrific],35.0,0.001693,993.0,0.048041,113.0,0.005467,1106.0,0.026754,-0.025060,-0.220285,-669.766337
lang [long],3.0,0.000145,69.0,0.003338,653.0,0.031592,722.0,0.017465,-0.017320,-0.159519,-557.344848
klein [small],5.0,0.000242,104.0,0.005031,546.0,0.026415,650.0,0.015723,-0.015481,-0.143616,-482.709249
informativ [informative],8.0,0.000387,45.0,0.002177,524.0,0.025351,569.0,0.013764,-0.013377,-0.125092,-397.139367
kindgerecht [suitable for children],3.0,0.000145,7.0,0.000339,408.0,0.019739,415.0,0.010039,-0.009894,-0.094212,-308.884563


In [22]:
fig = px.bar(
    diverging,
    x="literature_booktext_llr",
    y=diverging.index,
    hover_data = ['count_literature', 'count_booktext'],
    orientation="h"
)

fig.update_traces(marker_color="#333333")

fig.update_layout(
    height=600,
    width=800,
    xaxis_title=None,
    yaxis_title=None,
    showlegend=False,
    template="simple_white",
    margin=dict(l=120, r=40, t=40, b=80)  # larger top margin for annotations
)

# zero reference line
fig.add_vline(x=0, line_width=1.5, line_color="black")

# top annotations
fig.add_annotation(
    x=0.09, y=-0.14,
    xref="paper", yref="paper",
    text="more often used with<br>“Buch” [book] or “Text” [text]",
    showarrow=False,
    xanchor="left"
)

fig.add_annotation(
    x=0.89, y=-0.14,
    xref="paper", yref="paper",
    text="more often used with<br>“Literatur” [literature]",
    showarrow=False,
    xanchor="right"
)
fig.show()
fig.write_image("../results/literature_booktext_comparison.png", scale=4)